In [1]:
import pandas as pd
combined_df = pd.read_csv("raw_pr_dataset_full.csv")


# AI-related keywords
ai_keywords = [
    "chatgpt", "copilot", "gpt", "openai",
    "ai-generated", "generated by ai",
    "claude", "cursor", "llm"
]

def detect_ai(text):

    if pd.isna(text):
        return False

    text = str(text).lower()

    for word in ai_keywords:
        if word in text:
            return True

    return False


# detect AI keywords
combined_df["ai_title"] = combined_df["title"].apply(detect_ai)
combined_df["ai_body"] = combined_df["body"].apply(detect_ai)

# detect bots
combined_df["bot_user"] = combined_df["user"].str.contains("bot", case=False, na=False)

# final AI flag
combined_df["AI_flag"] = (
    combined_df["ai_title"] |
    combined_df["ai_body"] |
    combined_df["bot_user"]
)

# convert to 0/1
combined_df["AI_flag"] = combined_df["AI_flag"].astype(int)



In [2]:
combined_df["created_at"] = pd.to_datetime(combined_df["created_at"], format="ISO8601", errors="coerce")
combined_df["closed_at"] = pd.to_datetime(combined_df["closed_at"], format="ISO8601", errors="coerce")

combined_df["resolution_time_hours"] = (
    combined_df["closed_at"] - combined_df["created_at"]
).dt.total_seconds() / 3600


In [3]:
combined_df = combined_df[combined_df["resolution_time_hours"] >= 0.5]
combined_df = combined_df[combined_df["resolution_time_hours"] <= 720]

In [4]:
def classify_pr_type(text):
    if pd.isna(text):
        return "other"
    
    text = text.lower()
    
    if any(word in text for word in ["fix", "bug", "error", "issue", "crash", "fail", "failure", "incorrect", "handle"]):
        return "bug_fix"
    
    if any(word in text for word in ["add", "feature", "implement", "support", "enable", "introduce", "allow"]):
        return "feature"
    
    if any(word in text for word in ["refactor", "cleanup", "restructure", "simplify", "improve", "optimize", "rename"]):
        return "refactor"
    
    if any(word in text for word in ["doc", "readme", "documentation", "comment"]):
        return "documentation"
    
    if any(word in text for word in ["test", "unit test", "integration test", "e2e"]):
        return "test"
    
    if any(word in text for word in ["ci", "build", "pipeline", "dependency", "deps", "upgrade", "bump", "update"]):
        return "maintenance"
    
    return "other"

combined_df["combined_text"] = combined_df["title"].fillna("") + " " + combined_df["body"].fillna("")

combined_df["pr_type"] = combined_df["combined_text"].apply(classify_pr_type)

In [5]:
# Group by repository
repo_stats = combined_df.groupby("repo").agg(
    total_prs=("pr_number", "count"),
    ai_prs=("AI_flag", "sum")
).reset_index()

# Compute AI ratio
repo_stats["ai_ratio"] = repo_stats["ai_prs"] / repo_stats["total_prs"]

# Sort for inspection
repo_stats = repo_stats.sort_values(by="ai_ratio", ascending=False)

repo_stats.head(50)

,repo,total_prs,ai_prs,ai_ratio
18,go-vikunja/vikunja,365,300,0.821918
43,significant-gravitas/autogpt,1001,656,0.655345
36,pdfme/pdfme,489,277,0.566462
47,yamadashy/repomix,390,218,0.558974
13,dotnet/aspnetcore,1065,590,0.553991
17,glaredb/glaredb,1099,596,0.542311
21,jina-ai/node-deepresearch,30,16,0.533333
16,featureform/enrichmcp,66,35,0.530303
2,antiwork/gumroad,441,226,0.512472
27,microsoft/testfx,1167,567,0.485861


In [6]:
selected_repos_df = repo_stats[
    (repo_stats["ai_ratio"] >= 0.40) &
    (repo_stats["ai_ratio"] <= 0.60)
]

In [7]:
selected_repos = selected_repos_df["repo"].tolist()

df_selected = combined_df[combined_df["repo"].isin(selected_repos)]

In [8]:
df_selected.to_csv("dataset_preprocessed.csv", index=False)